# Lecture 2: Eigenvectors of the Three Standard Observables
### Computational Companion — States of a Two-Level System

This notebook verifies every claim in Lecture 2 with explicit Python computations:

1. **Pauli matrices and eigenvectors** — manual derivation vs `np.linalg.eigh`
2. **Spectral decomposition → expectation value** — two ways to compute ⟨M⟩
3. **Mutually unbiased bases (MUBs)** — full 6×6 overlap table
4. **Why ℂ not ℝ** — numerical proof that real coefficients can't produce a third symmetric pair
5. **Bloch sphere visualization** — all six cardinal states on the sphere
6. **Degrees of freedom** — parameterizing the state space with two real numbers

**Reference:** Lecture 2 notes ("Quantum Systems via Linear Algebra")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.linalg import expm

# ── Pauli matrices ──
I2 = np.eye(2, dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)

# ── Manual eigenvectors from Lecture 2 ──
# Z eigenvectors
e0 = np.array([[1], [0]], dtype=complex)
e1 = np.array([[0], [1]], dtype=complex)

# X eigenvectors
x_plus  = np.array([[1], [1]], dtype=complex) / np.sqrt(2)
x_minus = np.array([[1], [-1]], dtype=complex) / np.sqrt(2)

# Y eigenvectors
y_plus  = np.array([[1], [1j]], dtype=complex) / np.sqrt(2)
y_minus = np.array([[1], [-1j]], dtype=complex) / np.sqrt(2)

print("Pauli matrices and manual eigenvectors defined.")
print(f"  Z =\n{Z.real}\n")
print(f"  X =\n{X.real}\n")
print(f"  Y =\n{Y}")

## 1. Eigenvectors: Manual Derivation vs `np.linalg.eigh`

The lecture derives eigenvectors from symmetry constraints (equal measurement probabilities + orthogonality). Here we verify that those hand-derived vectors match what NumPy's eigensolver returns.

Key check: eigenvectors are unique only up to a **global phase** $e^{i\varphi}$, so we compare $|\langle \text{manual} | \text{library} \rangle| = 1$ rather than element-wise equality.

In [ ]:
# ── 1. Eigenvectors: manual vs np.linalg.eigh ──

manual_eigvecs = {
    "Z": {+1: e0, -1: e1},
    "X": {+1: x_plus, -1: x_minus},
    "Y": {+1: y_plus, -1: y_minus},
}
pauli = {"Z": Z, "X": X, "Y": Y}

print("=" * 65)
print("EIGENVECTOR COMPARISON: manual (Lecture 2) vs np.linalg.eigh")
print("=" * 65)

for name in ["Z", "X", "Y"]:
    evals, evecs = np.linalg.eigh(pauli[name])
    print(f"\nsigma_{name}  (eigenvalues from eigh: {evals})")
    for i, lam in enumerate(evals):
        lib_vec = evecs[:, i].reshape(-1, 1)
        manual_vec = manual_eigvecs[name][int(round(lam.real))]
        overlap = np.abs((manual_vec.conj().T @ lib_vec)[0, 0])
        # Also verify Mv = lambda*v directly
        residual = np.linalg.norm(pauli[name] @ manual_vec - lam * manual_vec)
        print(f"  λ={lam:+.0f}:  manual = {manual_vec.T[0]}  "
              f"|⟨manual|library⟩| = {overlap:.6f}  "
              f"‖Mv−λv‖ = {residual:.1e}")

## 2. Spectral Decomposition → Expectation Value (Two Ways)

The lecture derives that $\langle M \rangle = \mathbf{v}^\dagger M \mathbf{v} = \sum_i \lambda_i |\mathbf{e}_i^\dagger \mathbf{v}|^2$.

We verify both sides are equal for an arbitrary state and all three Pauli matrices.

In [ ]:
# ── 2. Expectation value: direct v†Mv vs spectral sum ──

# Arbitrary normalized state
psi = np.array([[3/5 + 4j/5], [1j/np.sqrt(2)]], dtype=complex)
psi = psi / np.linalg.norm(psi)

print("=" * 65)
print("EXPECTATION VALUE: v†Mv  vs  Σ λ_i |⟨e_i|v⟩|²")
print("=" * 65)
print(f"\nState |ψ⟩ = {psi.T[0]}\n")

for name in ["Z", "X", "Y"]:
    M = pauli[name]
    # Method 1: direct quadratic form
    exp_direct = (psi.conj().T @ M @ psi)[0, 0].real

    # Method 2: spectral decomposition sum
    evals, evecs = np.linalg.eigh(M)
    exp_spectral = sum(
        lam * np.abs((evecs[:, i].conj() @ psi.flatten()))**2
        for i, lam in enumerate(evals)
    )

    print(f"  ⟨σ_{name}⟩:  direct = {exp_direct:.6f}  "
          f"spectral = {exp_spectral.real:.6f}  "
          f"match = {np.isclose(exp_direct, exp_spectral.real)}")

# Also verify probabilities sum to 1 for each observable
print("\nProbability sums (must all be 1.0):")
for name in ["Z", "X", "Y"]:
    evals, evecs = np.linalg.eigh(pauli[name])
    total = sum(np.abs((evecs[:, i].conj() @ psi.flatten()))**2
                for i in range(len(evals)))
    print(f"  σ_{name}: Σ P(λ_i) = {total:.6f}")

## 3. Mutually Unbiased Bases — Full Overlap Table

The Z, X, Y eigenvectors form three **mutually unbiased bases (MUBs)**. The defining property: every vector in one basis has overlap $|\langle a | b \rangle|^2 = 1/d = 1/2$ with every vector in every *other* basis.

Within the same basis, eigenvectors are orthogonal ($|\langle a | b \rangle|^2 = 0$) or identical ($= 1$).

In [ ]:
# ── 3. MUB overlap table: |⟨a|b⟩|² for all 6 eigenvectors ──

all_vecs = {
    "Z+": e0, "Z−": e1,
    "X+": x_plus, "X−": x_minus,
    "Y+": y_plus, "Y−": y_minus,
}
labels = list(all_vecs.keys())
vecs = list(all_vecs.values())

n = len(labels)
overlap_sq = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        overlap_sq[i, j] = np.abs((vecs[i].conj().T @ vecs[j])[0, 0])**2

# Print table
print("=" * 65)
print("|⟨a|b⟩|²  OVERLAP TABLE  (MUB check)")
print("=" * 65)
print(f"{'':>5}", end="")
for l in labels:
    print(f"{l:>7}", end="")
print()
for i, l1 in enumerate(labels):
    print(f"{l1:>5}", end="")
    for j in range(n):
        print(f"{overlap_sq[i, j]:>7.2f}", end="")
    print()

print("\nWithin same observable: 1.00 (self) or 0.00 (orthogonal partner)")
print("Across different observables: 0.50 (mutually unbiased)")

# Heatmap visualization
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(overlap_sq, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(labels); ax.set_yticklabels(labels)
for i in range(n):
    for j in range(n):
        color = 'white' if overlap_sq[i, j] > 0.6 else 'black'
        ax.text(j, i, f"{overlap_sq[i, j]:.2f}", ha='center', va='center',
                fontsize=12, color=color)
# Draw boxes around each observable's 2x2 block
for start in [0, 2, 4]:
    rect = plt.Rectangle((start-0.5, start-0.5), 2, 2,
                          linewidth=2, edgecolor='blue', facecolor='none')
    ax.add_patch(rect)
ax.set_title("|⟨a|b⟩|² — Mutually Unbiased Bases\n"
             "(blue boxes = same observable; off-diagonal blocks = 0.50)")
plt.colorbar(im, ax=ax, label="|⟨a|b⟩|²")
plt.tight_layout()
plt.show()

## 4. Why ℂ, Not ℝ — The Third Basis Forces Complex Coefficients

The lecture proves that to find a third pair of vectors satisfying:
- $|α_0|^2 = |α_1|^2 = 1/2$ (equal Z-measurement probabilities)
- Orthogonal within the pair
- $|\langle x_± | w_+ \rangle|^2 = 1/2$ (equal X-measurement probabilities)

...the coefficient $b$ in $\mathbf{w}_+ = \frac{1}{\sqrt{2}}(\mathbf{e}_0 + b\,\mathbf{e}_1)$ must satisfy $|b|=1$ and $\text{Re}(b)=0$, forcing $b = \pm i$.

Here we sweep all unit-circle values of $b$ and show that only $b = \pm i$ satisfy all constraints simultaneously.

In [ ]:
# ── 4. Why ℂ not ℝ: sweep b on the unit circle ──

thetas = np.linspace(0, 2*np.pi, 500)

# For each b = e^{i*theta}, compute the three constraints
constraint_Z = []    # |b|^2 = 1  (always satisfied on unit circle)
constraint_orth = [] # w+†w- = 0 (always satisfied by construction w- = (e0 - b*e1)/sqrt(2))
constraint_X = []    # |x+† w+|^2 = 1/2

for theta in thetas:
    b = np.exp(1j * theta)
    w_plus = np.array([[1], [b]], dtype=complex) / np.sqrt(2)
    # Overlap with x_plus = (1,1)/sqrt(2)
    overlap_x = np.abs((x_plus.conj().T @ w_plus)[0, 0])**2
    constraint_X.append(overlap_x)

constraint_X = np.array(constraint_X)

# The constraint |x+†w+|² = 1/2 means |1+b|²/4 = 1/2, i.e. |1+b|² = 2
# Since b = e^{iθ}, |1+e^{iθ}|² = 2 + 2cos(θ) = 2 → cos(θ) = 0 → θ = π/2, 3π/2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: constraint_X vs theta
ax1.plot(np.degrees(thetas), constraint_X, 'steelblue', linewidth=2)
ax1.axhline(0.5, color='crimson', linestyle='--', linewidth=1.5, label='Target = 1/2')
ax1.axvline(90, color='green', linestyle=':', alpha=0.7, label='θ = 90° (b = i)')
ax1.axvline(270, color='green', linestyle=':', alpha=0.7, label='θ = 270° (b = −i)')
ax1.axvline(0, color='gray', linestyle=':', alpha=0.5, label='θ = 0° (b = 1, X eigvec)')
ax1.axvline(180, color='gray', linestyle=':', alpha=0.5, label='θ = 180° (b = −1, X eigvec)')
ax1.set_xlabel('θ (degrees) where b = e^{iθ}')
ax1.set_ylabel('|⟨x₊|w₊⟩|²')
ax1.set_title('X-measurement constraint: |⟨x₊|w₊⟩|² must equal 1/2')
ax1.legend(fontsize=8)
ax1.set_ylim(-0.05, 1.05)

# Right: unit circle with solutions highlighted
circle_x = np.cos(thetas)
circle_y = np.sin(thetas)
ax2.plot(circle_x, circle_y, 'lightgray', linewidth=1)
ax2.set_aspect('equal')

# Mark b = ±1 (X eigenvectors, real)
ax2.plot([1, -1], [0, 0], 'o', color='gray', markersize=10, label='b=±1 (X eigvecs, ℝ)')
# Mark b = ±i (Y eigenvectors, complex)
ax2.plot([0, 0], [1, -1], 's', color='green', markersize=12, label='b=±i (Y eigvecs, ℂ)')

ax2.annotate('b = 1\n(x₊)', (1, 0), textcoords="offset points", xytext=(10, -15), fontsize=9)
ax2.annotate('b = −1\n(x₋)', (-1, 0), textcoords="offset points", xytext=(-40, -15), fontsize=9)
ax2.annotate('b = i\n(y₊)', (0, 1), textcoords="offset points", xytext=(10, 5), fontsize=9, color='green')
ax2.annotate('b = −i\n(y₋)', (0, -1), textcoords="offset points", xytext=(10, -15), fontsize=9, color='green')

ax2.set_xlabel('Re(b)'); ax2.set_ylabel('Im(b)')
ax2.set_title('Unit circle: b = e^{iθ}\nOnly b = ±i gives a third MUB')
ax2.legend(fontsize=8, loc='lower right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Verify explicitly
for b_val, label in [(1, "b=1 (real)"), (-1, "b=-1 (real)"),
                      (1j, "b=i (complex)"), (-1j, "b=-i (complex)")]:
    w = np.array([[1], [b_val]], dtype=complex) / np.sqrt(2)
    overlap_xp = np.abs((x_plus.conj().T @ w)[0, 0])**2
    overlap_xm = np.abs((x_minus.conj().T @ w)[0, 0])**2
    overlap_zp = np.abs((e0.conj().T @ w)[0, 0])**2
    print(f"{label:>20}:  P(X=+1)={overlap_xp:.4f}  P(X=-1)={overlap_xm:.4f}  "
          f"P(Z=+1)={overlap_zp:.4f}  MUB? {np.isclose(overlap_xp, 0.5) and np.isclose(overlap_zp, 0.5)}")

## 5. Bloch Sphere Visualization — All Six Cardinal States

The two real degrees of freedom of a qubit state parameterize a point on the **Bloch sphere** $S^2$.

Any state $|\psi\rangle = \cos(\theta/2)|0\rangle + e^{i\phi}\sin(\theta/2)|1\rangle$ maps to the point $(\sin\theta\cos\phi,\; \sin\theta\sin\phi,\; \cos\theta)$.

The six eigenvectors of Z, X, Y sit at the six poles of the sphere: $\pm\hat{z}$, $\pm\hat{x}$, $\pm\hat{y}$.

In [ ]:
# ── 5. Bloch sphere with all six cardinal states ──

def state_to_bloch(state):
    """Convert a 2-component column vector to Bloch (x, y, z)."""
    alpha, beta = state[0, 0], state[1, 0]
    theta = np.pi if np.abs(alpha) < 1e-9 else 2 * np.arccos(np.clip(np.abs(alpha), 0, 1))
    phi = np.angle(beta) - np.angle(alpha)
    return np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)

def plot_bloch_sphere(ax, states, labels, colors, title=""):
    """Plot the Bloch sphere wireframe and state vectors."""
    u = np.linspace(0, 2*np.pi, 80)
    v = np.linspace(0, np.pi, 80)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones(np.size(u)), np.cos(v))
    ax.plot_surface(xs, ys, zs, color='lightcyan', alpha=0.08)
    ax.set_box_aspect([1, 1, 1])
    for axis in [([[-1.1,1.1],[0,0],[0,0]]),
                 ([[0,0],[-1.1,1.1],[0,0]]),
                 ([[0,0],[0,0],[-1.1,1.1]])]:
        ax.plot(axis[0], axis[1], axis[2], color='gray', linestyle='--', alpha=0.4)
    for i, state in enumerate(states):
        bx, by, bz = state_to_bloch(state)
        col = colors[i] if i < len(colors) else 'red'
        lbl = labels[i] if i < len(labels) else ''
        ax.quiver(0, 0, 0, bx, by, bz, color=col, length=1,
                  arrow_length_ratio=0.12, label=lbl, linewidth=2.5)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(title, fontsize=11)
    if labels:
        ax.legend(loc='upper left', fontsize=7)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

cardinal_states = [e0, e1, x_plus, x_minus, y_plus, y_minus]
cardinal_labels = ["|0⟩ (Z+)", "|1⟩ (Z−)", "|+⟩ (X+)", "|−⟩ (X−)", "|i⟩ (Y+)", "|−i⟩ (Y−)"]
cardinal_colors = ["blue", "navy", "red", "darkred", "green", "darkgreen"]

plot_bloch_sphere(ax, cardinal_states, cardinal_labels, cardinal_colors,
                  "Bloch Sphere: Six Cardinal States\n(Eigenvectors of Z, X, Y)")

# Verify Bloch coordinates
print("Bloch sphere coordinates of the six cardinal states:")
for lbl, vec in zip(cardinal_labels, cardinal_states):
    bx, by, bz = state_to_bloch(vec)
    print(f"  {lbl:>14}  →  ({bx:+.2f}, {by:+.2f}, {bz:+.2f})")

plt.tight_layout()
plt.show()

## 6. Degrees of Freedom — Parameterizing the State Space

A general qubit state has 4 real parameters (2 complex amplitudes). Normalization removes 1, global phase removes 1, leaving **2 real degrees of freedom**: $(\theta, \phi)$.

$$|\psi(\theta,\phi)\rangle = \cos\frac{\theta}{2}|0\rangle + e^{i\phi}\sin\frac{\theta}{2}|1\rangle, \quad \theta \in [0,\pi],\; \phi \in [0,2\pi)$$

We verify that multiplying by a global phase $e^{i\varphi}$ leaves all physical predictions unchanged.

In [ ]:
# ── 6. Degrees of freedom and global phase invariance ──

print("=" * 65)
print("DEGREES OF FREEDOM: θ, φ parameterization")
print("=" * 65)

# Sample some states on the Bloch sphere
test_states = [
    (np.pi/4, 0,       "θ=π/4, φ=0"),
    (np.pi/2, np.pi/3, "θ=π/2, φ=π/3"),
    (np.pi/3, np.pi,   "θ=π/3, φ=π"),
    (2*np.pi/3, 5*np.pi/4, "θ=2π/3, φ=5π/4"),
]

for theta, phi, desc in test_states:
    psi_param = np.array([[np.cos(theta/2)],
                          [np.exp(1j*phi) * np.sin(theta/2)]], dtype=complex)
    bx, by, bz = state_to_bloch(psi_param)
    # Expectation values should match Bloch coordinates
    ex = (psi_param.conj().T @ X @ psi_param)[0, 0].real
    ey = (psi_param.conj().T @ Y @ psi_param)[0, 0].real
    ez = (psi_param.conj().T @ Z @ psi_param)[0, 0].real
    print(f"  {desc:>22}:  Bloch=({bx:+.3f},{by:+.3f},{bz:+.3f})  "
          f"⟨σ⟩=({ex:+.3f},{ey:+.3f},{ez:+.3f})  match={np.allclose([bx,by,bz],[ex,ey,ez])}")

# Global phase invariance verification
print("\n" + "=" * 65)
print("GLOBAL PHASE INVARIANCE")
print("=" * 65)

psi_test = np.array([[np.cos(np.pi/6)],
                      [np.exp(1j*np.pi/5) * np.sin(np.pi/6)]], dtype=complex)
phases = [0, np.pi/7, np.pi/3, np.pi, 5*np.pi/3]

print(f"\n|ψ⟩ = {psi_test.T[0]}")
print(f"{'phase':>10} {'P(Z=+1)':>10} {'P(Z=-1)':>10} {'⟨X⟩':>10} {'⟨Y⟩':>10} {'⟨Z⟩':>10}")
for phi_g in phases:
    rotated = np.exp(1j * phi_g) * psi_test
    pz_p = np.abs((e0.conj().T @ rotated)[0, 0])**2
    pz_m = np.abs((e1.conj().T @ rotated)[0, 0])**2
    ex = (rotated.conj().T @ X @ rotated)[0, 0].real
    ey = (rotated.conj().T @ Y @ rotated)[0, 0].real
    ez = (rotated.conj().T @ Z @ rotated)[0, 0].real
    print(f"{phi_g:>10.4f} {pz_p:>10.6f} {pz_m:>10.6f} {ex:>10.6f} {ey:>10.6f} {ez:>10.6f}")
print("\nAll rows identical ⇒ global phase has no physical effect.")

## 7. Matrix Verification — $M \mathbf{v} = \lambda \mathbf{v}$ for All Six Eigenvectors

The lecture derives eigenvectors from symmetry constraints alone. Here we close the loop by verifying that each hand-derived eigenvector satisfies $M \mathbf{v} = \lambda \mathbf{v}$ for the corresponding Pauli matrix, and that reverse-engineering the matrix from its eigendecomposition recovers the original Pauli matrix:

$$M = \sum_i \lambda_i \mathbf{e}_i \mathbf{e}_i^\dagger$$

In [ ]:
# ── 7. Matrix verification: Mv = λv and eigendecomposition reconstruction ──

print("=" * 65)
print("MATRIX VERIFICATION: Mv = λv for all six eigenvectors")
print("=" * 65)

checks = [
    ("Z", Z, "+1", +1, e0), ("Z", Z, "−1", -1, e1),
    ("X", X, "+1", +1, x_plus), ("X", X, "−1", -1, x_minus),
    ("Y", Y, "+1", +1, y_plus), ("Y", Y, "−1", -1, y_minus),
]

for name, M, lam_str, lam, vec in checks:
    Mv = M @ vec
    lam_v = lam * vec
    residual = np.linalg.norm(Mv - lam_v)
    print(f"  {name}·{name.lower()}{lam_str} − ({lam_str})·{name.lower()}{lam_str} :  "
          f"‖Mv − λv‖ = {residual:.1e}  ✓" if residual < 1e-14 else f"  FAIL: {residual:.1e}")

# Reconstruct each Pauli matrix from its eigendecomposition
print("\n" + "=" * 65)
print("RECONSTRUCTION: M = Σ λ_i |e_i⟩⟨e_i|")
print("=" * 65)

for name, M, vecs_pairs in [("Z", Z, [(+1, e0), (-1, e1)]),
                              ("X", X, [(+1, x_plus), (-1, x_minus)]),
                              ("Y", Y, [(+1, y_plus), (-1, y_minus)])]:
    M_reconstructed = sum(lam * (v @ v.conj().T) for lam, v in vecs_pairs)
    err = np.linalg.norm(M - M_reconstructed)
    print(f"  {name}: ‖M − Σλ|e⟩⟨e|‖ = {err:.1e}  match = {np.allclose(M, M_reconstructed)}")
    if name == "Y":
        print(f"\n  Y reconstructed =\n{M_reconstructed}")
        print(f"  Y original      =\n{Y}")

## 8. Commutator Relations and Non-Commutativity

The three Pauli matrices satisfy the cyclic commutation relations:
$$[Z,X] = 2iY, \quad [X,Y] = 2iZ, \quad [Y,Z] = 2iX$$

Non-commutativity is the algebraic reason that measuring one observable randomizes the outcome of another. We also verify that each Pauli matrix squares to the identity ($\sigma_k^2 = I$) and compute the anticommutators ($\{\sigma_i, \sigma_j\} = 2\delta_{ij}I$).

In [ ]:
# ── 8. Commutator relations and algebraic identities ──

print("=" * 65)
print("CYCLIC COMMUTATOR RELATIONS: [σ_i, σ_j] = 2iε_ijk σ_k")
print("=" * 65)

comm_pairs = [
    ("Z", "X", Z, X, "Y", Y),
    ("X", "Y", X, Y, "Z", Z),
    ("Y", "Z", Y, Z, "X", X),
]

for n1, n2, M1, M2, n3, M3 in comm_pairs:
    comm = M1 @ M2 - M2 @ M1
    expected = 2j * M3
    print(f"\n[{n1}, {n2}] =\n{comm}")
    print(f"2i·{n3}   =\n{expected}")
    print(f"Match: {np.allclose(comm, expected)}")

# Pauli algebra: σ_k² = I
print("\n" + "=" * 65)
print("PAULI ALGEBRA: σ_k² = I")
print("=" * 65)
for name, M in [("Z", Z), ("X", X), ("Y", Y)]:
    sq = M @ M
    print(f"  {name}² = I ? {np.allclose(sq, I2)}")

# Anticommutators: {σ_i, σ_j} = σ_iσ_j + σ_jσ_i = 2δ_ij I
print("\n" + "=" * 65)
print("ANTICOMMUTATORS: {σ_i, σ_j} = 2δ_ij I")
print("=" * 65)
paulis = {"Z": Z, "X": X, "Y": Y}
for n1 in ["Z", "X", "Y"]:
    for n2 in ["Z", "X", "Y"]:
        anticomm = paulis[n1] @ paulis[n2] + paulis[n2] @ paulis[n1]
        expected_val = 2 * I2 if n1 == n2 else np.zeros((2, 2))
        status = "2I" if n1 == n2 else "0"
        print(f"  {{{n1},{n2}}} = {status}  ✓" if np.allclose(anticomm, expected_val) else f"  {{{n1},{n2}}} FAIL")

## 9. Born-Rule Probabilities for a General State

Given an arbitrary state $|\psi\rangle$, compute the measurement probabilities for all three observables and verify they sum to 1. Also verify the reciprocal relationship: expressing Z-eigenvectors in the X-basis and vice versa.

In [ ]:
# ── 9. Born-rule probabilities for a general state ──

print("=" * 65)
print("BORN-RULE PROBABILITIES FOR A GENERAL STATE")
print("=" * 65)

# General state: |ψ⟩ = cos(π/5)|0⟩ + e^{iπ/3}sin(π/5)|1⟩
theta_gen, phi_gen = np.pi/5, np.pi/3
psi_gen = np.array([[np.cos(theta_gen/2)],
                     [np.exp(1j*phi_gen) * np.sin(theta_gen/2)]], dtype=complex)
print(f"\n|ψ⟩ = {psi_gen.T[0]}  (θ={np.degrees(theta_gen):.0f}°, φ={np.degrees(phi_gen):.0f}°)\n")

for obs_name, eigvecs in [("Z", [("Z+", e0), ("Z−", e1)]),
                            ("X", [("X+", x_plus), ("X−", x_minus)]),
                            ("Y", [("Y+", y_plus), ("Y−", y_minus)])]:
    probs = []
    for lbl, ev in eigvecs:
        p = np.abs((ev.conj().T @ psi_gen)[0, 0])**2
        probs.append(p)
        print(f"  P({lbl}) = |⟨{lbl}|ψ⟩|² = {p:.6f}")
    print(f"  Sum = {sum(probs):.6f}\n")

# Reciprocal relationship: Z-eigenvectors in X-basis
print("=" * 65)
print("RECIPROCAL RELATIONSHIP: Z-eigvecs in X-basis")
print("=" * 65)

for name, vec in [("e0", e0), ("e1", e1)]:
    c_xp = (x_plus.conj().T @ vec)[0, 0]
    c_xm = (x_minus.conj().T @ vec)[0, 0]
    print(f"\n  {name} = ({c_xp:.4f})|x+⟩ + ({c_xm:.4f})|x−⟩")
    print(f"  |coefficients|² = {np.abs(c_xp)**2:.4f}, {np.abs(c_xm)**2:.4f}  (both 1/2)")

# Y-eigenvectors in X-basis (the key check from §3.3)
print("\n" + "=" * 65)
print("Y-EIGVECS IN X-BASIS (verify |(1±i)/2|² = 1/2)")
print("=" * 65)
for name, vec in [("y+", y_plus), ("y−", y_minus)]:
    c_xp = (x_plus.conj().T @ vec)[0, 0]
    c_xm = (x_minus.conj().T @ vec)[0, 0]
    print(f"\n  {name} = ({c_xp:.4f})|x+⟩ + ({c_xm:.4f})|x−⟩")
    print(f"  |coefficients|² = {np.abs(c_xp)**2:.4f}, {np.abs(c_xm)**2:.4f}  (both 1/2)")

## 10. State Collapse and Sequential Measurements

Demonstrates the key non-classical feature: measuring one observable randomizes the outcome of a non-commuting observable.

- **Path A**: Measure Z → get +1 → measure Z again → still +1 (deterministic)
- **Path B**: Measure Z → get +1 → measure X → get ±1 with prob 1/2 → measure Z → now random

This is a direct consequence of $[Z, X] \neq 0$.

In [ ]:
# ── 10. State collapse and sequential measurements ──

print("=" * 65)
print("SEQUENTIAL MEASUREMENTS: non-commutativity in action")
print("=" * 65)

# Start in |0⟩ (Z eigenstate, eigenvalue +1)
state = e0.copy()
print(f"\nInitial state: |0⟩ = {state.T[0]}")

# PATH A: Measure Z → measure Z again
print("\n── Path A: Z then Z ──")
p_z_plus = np.abs((e0.conj().T @ state)[0, 0])**2
print(f"  Measure Z: P(+1) = {p_z_plus:.4f} → get +1, collapse to |0⟩")
state_after_z = e0.copy()
p_z_plus_again = np.abs((e0.conj().T @ state_after_z)[0, 0])**2
print(f"  Measure Z again: P(+1) = {p_z_plus_again:.4f} → deterministic!")

# PATH B: Measure Z → measure X → measure Z
print("\n── Path B: Z then X then Z ──")
state = e0.copy()
p_z_plus = np.abs((e0.conj().T @ state)[0, 0])**2
print(f"  Measure Z: P(+1) = {p_z_plus:.4f} → get +1, collapse to |0⟩")

# Now measure X on |0⟩
p_x_plus  = np.abs((x_plus.conj().T @ e0)[0, 0])**2
p_x_minus = np.abs((x_minus.conj().T @ e0)[0, 0])**2
print(f"  Measure X: P(+1) = {p_x_plus:.4f}, P(−1) = {p_x_minus:.4f}")
print(f"  Suppose we get +1, collapse to |+⟩ = {x_plus.T[0]}")

# Now measure Z on |+⟩
state_after_x = x_plus.copy()
p_z_plus_final  = np.abs((e0.conj().T @ state_after_x)[0, 0])**2
p_z_minus_final = np.abs((e1.conj().T @ state_after_x)[0, 0])**2
print(f"  Measure Z: P(+1) = {p_z_plus_final:.4f}, P(−1) = {p_z_minus_final:.4f} → randomized!")
print(f"\n  The X measurement destroyed the definite Z value.")
print(f"  This happens because [Z, X] ≠ 0:")
print(f"  [Z, X] =\n{Z @ X - X @ Z}")

# Monte Carlo simulation of Path B
print("\n── Monte Carlo: 10000 runs of Path B ──")
rng = np.random.default_rng(42)
N = 10000
z_results_after_x = []
for _ in range(N):
    # Start in |0⟩, measure X
    if rng.random() < p_x_plus:
        collapsed = x_plus
    else:
        collapsed = x_minus
    # Measure Z on the X-collapsed state
    p_z = np.abs((e0.conj().T @ collapsed)[0, 0])**2
    z_results_after_x.append(1 if rng.random() < p_z else -1)

z_results = np.array(z_results_after_x)
print(f"  P(Z=+1) observed = {np.mean(z_results == 1):.4f}  (theory: 0.5)")
print(f"  P(Z=−1) observed = {np.mean(z_results == -1):.4f}  (theory: 0.5)")

## 11. Visualizing the State Space — Bloch Sphere Sampling

Every qubit state maps to a unique point on the Bloch sphere. Here we:
1. Sample random states and plot them on the sphere
2. Show that the expectation value vector $(\langle X \rangle, \langle Y \rangle, \langle Z \rangle)$ always lies on the unit sphere for pure states
3. Demonstrate the (θ, φ) → Bloch mapping exhaustively

In [ ]:
# ── 11. Bloch sphere sampling and (θ,φ) exhaustive mapping ──

# Sample random pure states and verify they all land on the unit sphere
rng = np.random.default_rng(123)
N_samples = 500
bloch_points = []

for _ in range(N_samples):
    # Random state via Haar measure: random complex vector, then normalize
    raw = rng.standard_normal(2) + 1j * rng.standard_normal(2)
    psi_rand = raw.reshape(-1, 1) / np.linalg.norm(raw)
    bx, by, bz = state_to_bloch(psi_rand)
    bloch_points.append((bx, by, bz))

bx_arr, by_arr, bz_arr = zip(*bloch_points)
radii = np.sqrt(np.array(bx_arr)**2 + np.array(by_arr)**2 + np.array(bz_arr)**2)
print(f"Sampled {N_samples} random pure states.")
print(f"Bloch vector radii: min={radii.min():.6f}, max={radii.max():.6f}, mean={radii.mean():.6f}")
print("All lie on the unit sphere (radius = 1) ✓")

# Exhaustive (θ,φ) grid
theta_grid = np.linspace(0, np.pi, 30)
phi_grid = np.linspace(0, 2*np.pi, 60)
grid_x, grid_y, grid_z = [], [], []
for th in theta_grid:
    for ph in phi_grid:
        grid_x.append(np.sin(th)*np.cos(ph))
        grid_y.append(np.sin(th)*np.sin(ph))
        grid_z.append(np.cos(th))

fig = plt.figure(figsize=(14, 6))

# Left: random samples
ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(bx_arr, by_arr, bz_arr, c=bz_arr, cmap='coolwarm', s=8, alpha=0.6)
ax1.set_box_aspect([1,1,1]); ax1.set_xlabel('X'); ax1.set_ylabel('Y'); ax1.set_zlabel('Z')
ax1.set_title(f'{N_samples} Random Pure States\non the Bloch Sphere')

# Right: (θ,φ) grid
ax2 = fig.add_subplot(122, projection='3d')
ax2.scatter(grid_x, grid_y, grid_z, c=grid_z, cmap='coolwarm', s=5, alpha=0.5)
# Highlight cardinal states
for lbl, vec, col in [("Z+", e0, 'blue'), ("Z−", e1, 'navy'),
                       ("X+", x_plus, 'red'), ("X−", x_minus, 'darkred'),
                       ("Y+", y_plus, 'green'), ("Y−", y_minus, 'darkgreen')]:
    bx_c, by_c, bz_c = state_to_bloch(vec)
    ax2.scatter([bx_c], [by_c], [bz_c], s=100, marker='*', c=col, edgecolors='black', zorder=5)
    ax2.text(bx_c*1.15, by_c*1.15, bz_c*1.15, lbl, fontsize=8)
ax2.set_box_aspect([1,1,1]); ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')
ax2.set_title('(θ,φ) Grid → Bloch Sphere\n★ = Cardinal States')

plt.tight_layout()
plt.show()

# Lecture 2: Eigenvectors of the Three Standard Observables
### Computational Companion — States of a Two-Level System

This notebook verifies every claim in Lecture 2 with explicit Python computations:

1. **Pauli matrices and eigenvectors** — manual derivation vs `np.linalg.eigh`
2. **Spectral decomposition → expectation value** — two ways to compute ⟨M⟩
3. **Mutually unbiased bases (MUBs)** — full 6×6 overlap table
4. **Why ℂ not ℝ** — numerical proof that real coefficients can't produce a third symmetric pair
5. **Bloch sphere visualization** — all six cardinal states on the sphere
6. **Degrees of freedom** — parameterizing the state space with two real numbers

**Reference:** Lecture 2 notes ("Quantum Systems via Linear Algebra")